# Fine-tuning Moonshine ASR — język polski

Notebook do fine-tuningu modelu [Moonshine](https://github.com/usefulsensors/moonshine) na języku polskim (dataset MLS Polish).

**Wymagania:**
- Runtime: GPU (T4 lub lepszy — zmień w: Runtime → Change runtime type → T4 GPU)
- Czas: ~2–4 h dla `tiny`, ~4–8 h dla `base`
- HuggingFace token: opcjonalnie (do pushowania modelu)

**Dane:** [facebook/multilingual_librispeech](https://huggingface.co/datasets/facebook/multilingual_librispeech) — język `polish` (~25h)

## 0. Konfiguracja

Ustaw poniższe zmienne przed uruchomieniem notebooka.

In [ ]:
# ============================================================
# KONFIGURACJA — zmień według potrzeb
# ============================================================

# Model: "tiny" (27M params, szybszy) lub "base" (61M params, dokładniejszy)
MODEL_SIZE = "tiny"   # "tiny" | "base"

# Czy zapisywać checkpointy na Google Drive? (zalecane — sesja Colab może się rozłączyć)
USE_GOOGLE_DRIVE = True

# Ścieżka na Drive do zapisu wyników
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/moonshine-polish"

# Test mode: trenuj tylko na 100 próbkach (~5 minut) żeby sprawdzić czy wszystko działa
TEST_MODE = True   # Zmień na False dla pełnego treningu

# HuggingFace token (opcjonalnie — do push_to_hub)
# Zostaw pusty jeśli nie chcesz pushować
HF_TOKEN = ""

# HuggingFace repo do którego pushować (np. "twoja-nazwa/moonshine-pl-tiny")
HF_REPO_ID = ""

# ============================================================
print(f"Model: moonshine-{MODEL_SIZE}")
print(f"Test mode: {TEST_MODE}")
print(f"Google Drive: {USE_GOOGLE_DRIVE}")

## 1. Sprawdzenie GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Brak GPU! Zmień runtime: Runtime → Change runtime type → T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")

## 2. Google Drive (opcjonalnie)

Montowanie Drive chroni checkpointy przed utratą przy rozłączeniu sesji.

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f"Drive zamontowany. Wyniki będą zapisywane w: {DRIVE_OUTPUT_DIR}")
else:
    print("Używam lokalnego storage Colab (wyniki przepadną po rozłączeniu!)")

## 3. Instalacja zależności i klonowanie repozytorium

In [ ]:
import os

REPO_DIR = "/content/finetune-moonshine-asr"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ChrisTyb77/finetune-moonshine-asr.git {REPO_DIR}
else:
    print("Repozytorium już sklonowane, pomijam.")

%cd {REPO_DIR}

In [ ]:
# Instalacja zależności
!pip install -q -r requirements.txt
print("Instalacja zakończona.")

In [ ]:
# Weryfikacja kluczowych paczek
import transformers, datasets, evaluate, accelerate
print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")

## 4. HuggingFace login (opcjonalnie)

In [ ]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Zalogowano do HuggingFace Hub.")
else:
    print("Pominięto logowanie (brak HF_TOKEN).")

## 5. Przygotowanie konfiguracji

Wybieramy config odpowiedni dla wybranego modelu i ustawiamy ścieżkę wyjściową.

In [ ]:
import yaml
import shutil

# Wybierz bazowy config
BASE_CONFIG = f"configs/mls_polish_{MODEL_SIZE}.yaml"

# Ścieżka wyjściowa (Drive lub lokalnie)
OUTPUT_DIR = (
    f"{DRIVE_OUTPUT_DIR}/results-moonshine-pl-{MODEL_SIZE}"
    if USE_GOOGLE_DRIVE
    else f"./results-moonshine-pl-{MODEL_SIZE}"
)

# Wczytaj config i nadpisz output_dir oraz logging_dir
with open(BASE_CONFIG, "r") as f:
    config = yaml.safe_load(f)

config["training"]["output_dir"] = OUTPUT_DIR
config["training"]["logging_dir"] = OUTPUT_DIR + "/logs"

# Opcjonalnie: push do Hub
if HF_TOKEN and HF_REPO_ID:
    config["training"]["push_to_hub"] = True
    config["training"]["hub_model_id"] = HF_REPO_ID
    config["training"]["hub_token"] = HF_TOKEN

# Zapisz zmodyfikowany config
RUN_CONFIG = f"/tmp/mls_polish_{MODEL_SIZE}_run.yaml"
with open(RUN_CONFIG, "w") as f:
    yaml.dump(config, f, allow_unicode=True)

print(f"Config: {BASE_CONFIG}")
print(f"Output: {OUTPUT_DIR}")
print(f"Run config: {RUN_CONFIG}")

## 6. Test mode — szybka weryfikacja

Uruchom ten krok żeby sprawdzić, czy wszystko działa (100 próbek, ~5 minut).  
Po udanym teście zmień `TEST_MODE = False` w komórce konfiguracyjnej i przejdź do kroku 7.

In [ ]:
if TEST_MODE:
    print("Uruchamianie w trybie testowym (100 próbek)...")
    !python train.py \
        --config {RUN_CONFIG} \
        --no-curriculum \
        --test-mode
else:
    print("TEST_MODE=False — pomiń tę komórkę i przejdź do kroku 7.")

## 7. Pełny trening

Uruchom po udanym teście (`TEST_MODE = False`).

Szacowany czas:
- `tiny` na T4: ~2–3 h
- `base` na T4: ~4–6 h

Checkpointy są zapisywane co 250 kroków — możesz wznowić trening po rozłączeniu.

In [ ]:
if not TEST_MODE:
    print(f"Trening moonshine-{MODEL_SIZE} na MLS Polish...")
    !python train.py \
        --config {RUN_CONFIG} \
        --no-curriculum
else:
    print("TEST_MODE=True — zmień na False w komórce konfiguracyjnej żeby uruchomić pełny trening.")

### Wznowienie po rozłączeniu

Jeśli sesja Colab się rozłączyła, uruchom ponownie komórki 1–5, a następnie tę komórkę:

In [ ]:
# Wznowienie treningu od ostatniego checkpointu
# Odkomentuj i dostosuj ścieżkę do ostatniego checkpointu

# LAST_CHECKPOINT = f"{OUTPUT_DIR}/checkpoint-1500"   # <-- zmień numer

# !python train.py \
#     --config {RUN_CONFIG} \
#     --no-curriculum \
#     --resume {LAST_CHECKPOINT}

## 8. Monitorowanie treningu (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/logs

## 9. Ewaluacja (WER)

Oblicz Word Error Rate na zbiorze testowym MLS Polish.

In [ ]:
import os

BEST_CHECKPOINT = os.path.join(OUTPUT_DIR, "checkpoint-best")

# Jeśli checkpoint-best nie istnieje, użyj katalogu final
if not os.path.exists(BEST_CHECKPOINT):
    BEST_CHECKPOINT = os.path.join(OUTPUT_DIR, "final")

print(f"Ewaluacja checkpointu: {BEST_CHECKPOINT}")

!python scripts/evaluate.py \
    --model {BEST_CHECKPOINT} \
    --dataset facebook/multilingual_librispeech \
    --language polish \
    --split test

## 10. Demo — transkrypcja przykładowego audio

In [ ]:
from datasets import load_dataset, Audio
from transformers import AutoProcessor, MoonshineForConditionalGeneration
import torch

# Wczytaj próbkę z MLS Polish (test split)
print("Wczytywanie próbki z MLS Polish...")
sample_ds = load_dataset(
    "facebook/multilingual_librispeech",
    "polish",
    split="test",
    streaming=True
)
sample = next(iter(sample_ds))

# Wczytaj model i processor
processor = AutoProcessor.from_pretrained(BEST_CHECKPOINT)
model = MoonshineForConditionalGeneration.from_pretrained(BEST_CHECKPOINT)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Resampling do 16kHz jeśli potrzebny
audio_array = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

if sr != 16000:
    import librosa
    audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)

# Transkrypcja
inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    generated = model.generate(
        input_values=inputs["input_values"],
        max_new_tokens=80,
        num_beams=5,
        repetition_penalty=1.3,
    )

transcription = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

print(f"\nReferencja:    {sample['transcript']}")
print(f"Transkrypcja:  {transcription}")

## 11. Eksport do ONNX (streaming)

Konwertuje wytrenowany model do formatu ONNX gotowego do inference na CPU.

In [ ]:
ONNX_OUTPUT = os.path.join(OUTPUT_DIR, f"moonshine-pl-{MODEL_SIZE}-onnx")

!python scripts/convert_for_deployment.py \
    --model {BEST_CHECKPOINT} \
    --output {ONNX_OUTPUT}

print(f"\nModel ONNX zapisany w: {ONNX_OUTPUT}")

## 12. Push do HuggingFace Hub (opcjonalnie)

Wymaga ustawionego `HF_TOKEN` i `HF_REPO_ID` w komórce konfiguracyjnej.

In [ ]:
if HF_TOKEN and HF_REPO_ID:
    from transformers import AutoProcessor, MoonshineForConditionalGeneration

    print(f"Pushowanie modelu do: {HF_REPO_ID}")

    push_model = MoonshineForConditionalGeneration.from_pretrained(BEST_CHECKPOINT)
    push_processor = AutoProcessor.from_pretrained(BEST_CHECKPOINT)

    push_model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    push_processor.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

    print(f"Model dostępny na: https://huggingface.co/{HF_REPO_ID}")
else:
    print("Pominięto (brak HF_TOKEN lub HF_REPO_ID).")